# TP1 - Démarrage

1. Constitution des équipes (2-3) et création du dépôt Git — arborescence imposée : data/ , notebooks/ , src/ , docs/ .

✅

2. Aller chercher les données vous-mêmes sur world.openfoodfacts.org/data : explorer les formats proposés,
choisir celui qui tient sur votre machine, le télécharger et documenter votre choix dans docs/journal.md (format retenu, taille, date de l'export, pourquoi celui-là).

✅
format retenu : .parquet - 7.7Go

3. Premier notebook pandas : chargement, dimensions, types, mémoire, taux de remplissage par colonne.

In [1]:
# Commencer par faire les import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
PATH = "../../data/food.parquet"
AXES=[
    'code',
    'brands',
    'countries_tags',
    'nutriscore_grade'
]

In [3]:
df_brut = pd.read_parquet(PATH, columns=AXES)

In [4]:
df_brut.shape

(4636471, 4)

In [5]:
df_brut.head()

,code,brands,countries_tags,nutriscore_grade
0,0000101209159,Bovetti,[en:france],e
1,0000105000011,Lagg's,[en:united-states],unknown
2,0000105000042,Lagg's,[en:united-states],unknown
3,0000105000059,Lagg's,[en:united-states],unknown
4,0000105000073,Lagg's,[en:united-states],unknown


In [6]:
df_brut.dtypes

code                   str
brands                 str
countries_tags      object
nutriscore_grade       str
dtype: object

In [7]:
memory = df_brut.memory_usage(deep=True).sum() / 1_000_000 # Mettre en Mo
print(f"Mémoire utilisée : {memory} Mo")

Mémoire utilisée : 783.222404 Mo


In [8]:
columns_completion = df_brut.notna().mean(axis=0) * 100
columns_completion.round(2).sort_values()

# brands               65.60
# nutriscore_grade     99.06
# countries_tags       99.79
# code                100.00
# dtype: float64

brands               65.60
nutriscore_grade     99.06
countries_tags       99.79
code                100.00
dtype: float64

In [9]:
df_brut['nutriscore_grade'].value_counts(dropna=False)

# nutriscore_grade
# unknown           3110742
# e                  387545
# d                  346120
# c                  287328
# a                  200871
# b                  159205
# not-applicable     101213
# NaN                 43447
# Name: count, dtype: int64

nutriscore_grade
unknown           3110742
e                  387545
d                  346120
c                  287328
a                  200871
b                  159205
not-applicable     101213
NaN                 43447
Name: count, dtype: int64

4. Cinq questions à résoudre en équipe : combien de produits vendus en France ? quelle part a un Nutri-Score renseigné ? les dix marques les plus présentes ? le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g ) ? qu'est-ce qui vous semble le plus « sale » dans ces données ?

### Q1 - combien de produits vendus en France ?

In [10]:
is_france = df_brut['countries_tags'].apply(lambda x : x is not None and 'en:france' in x)
french_products_count = is_france.sum()

In [11]:
print(f"Nombre de produits vendus en France : {french_products_count}")
# Nombre de produits vendus en France : 1247336

Nombre de produits vendus en France : 1247336


### Q2 - quelle part a un Nutri-Score renseigné ?

In [12]:
df_fr = df_brut[is_france].copy()

In [13]:
df_fr['nutriscore_grade'] = df_fr['nutriscore_grade'].replace('unknown', np.nan)

#### Part des NS renseignés

In [14]:
round(df_fr['nutriscore_grade'].value_counts(normalize=True) * 100, 2)

# nutriscore_grade
# e                 26.18
# d                 24.28
# c                 19.10
# a                 12.51
# b                  9.82
# not-applicable     8.11
# Name: proportion, dtype: float64

nutriscore_grade
e                 26.18
d                 24.28
c                 19.10
a                 12.51
b                  9.82
not-applicable     8.11
Name: proportion, dtype: float64

#### Part des NS renseignés avec les NaN

In [15]:
round(df_fr['nutriscore_grade'].value_counts(normalize=True, dropna=False) * 100, 2)

# nutriscore_grade
# NaN               59.54
# e                 10.59
# d                  9.82
# c                  7.73
# a                  5.06
# b                  3.97
# not-applicable     3.28
# Name: proportion, dtype: float64

nutriscore_grade
NaN               59.54
e                 10.59
d                  9.82
c                  7.73
a                  5.06
b                  3.97
not-applicable     3.28
Name: proportion, dtype: float64

In [16]:
round(df_fr['nutriscore_grade'].isna().mean() * 100, 2) # 59.54 %

np.float64(59.54)

### Q3 - les dix marques les plus présentes ?

In [17]:
brands = df_fr['brands']

In [18]:
brands.value_counts().sort_values(ascending=False).head(10)

# brands
#                 54688
# Carrefour       12052
# U               11974
# Auchan           6291
# Leader Price     5427
# Casino           5119
# Cora             3960
# Le Gaulois       3548
# Picard           3501
# Monoprix         3399
# Name: count, dtype: int64

brands
                54688
Carrefour       12052
U               11974
Auchan           6291
Leader Price     5427
Casino           5119
Cora             3960
Le Gaulois       3548
Picard           3501
Monoprix         3399
Name: count, dtype: int64

### Q4 - le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g ) ? 

In [19]:
df_nutriments = pd.read_parquet(PATH, columns=["nutriments"]) # ~ 1min
nutriments_fr = df_nutriments.loc[is_france, 'nutriments'].copy()

In [27]:
df_nutriments['nutriments'][0]

array([{'name': 'fruits-vegetables-nuts', 'value': None, '100g': 68.38999938964844, 'serving': None, 'unit': '%', 'prepared_value': None, 'prepared_100g': None, 'prepared_serving': None, 'prepared_unit': None},
       {'name': 'carbohydrates', 'value': None, '100g': 36.0, 'serving': None, 'unit': 'g', 'prepared_value': None, 'prepared_100g': None, 'prepared_serving': None, 'prepared_unit': None},
       {'name': 'proteins', 'value': None, '100g': 8.0, 'serving': None, 'unit': 'g', 'prepared_value': None, 'prepared_100g': None, 'prepared_serving': None, 'prepared_unit': None},
       {'name': 'saturated-fat', 'value': None, '100g': 10.0, 'serving': None, 'unit': 'g', 'prepared_value': None, 'prepared_100g': None, 'prepared_serving': None, 'prepared_unit': None},
       {'name': 'added-sugars', 'value': None, '100g': 14.720000267028809, 'serving': None, 'unit': 'g', 'prepared_value': None, 'prepared_100g': None, 'prepared_serving': None, 'prepared_unit': None},
       {'name': 'sodium', 

In [20]:
def get_100g(nutrients, name):
    if nutrients is None:
        return np.nan
    for nutrient in nutrients:
        if nutrient['name'] == name :
            return nutrient['100g']
    return np.nan

In [22]:
for name in ["energy", "sugars", "salt"]:
    values = nutriments_fr.apply(lambda nutrients: get_100g(nutrients, name))
    print(f"{name} : {values.isna().mean() * 100:.2f} % manquants")

energy : 28.79 % manquants
sugars : 29.39 % manquants
salt : 33.74 % manquants


### Q5 - qu'est-ce qui vous semble le plus « sale » dans ces données ?


5. Restitution : dix minutes par équipe en fin de journée — dont la stratégie de téléchargement retenue, qui diffère sûrement d'une équipe à l'autre.
À committer : dépôt initialisé + notebook + réponses dans docs/journal.md . Les données brutes restent hors du dépôt — seul le script ou la commande qui les récupère est versionné.

✅